# Best-Seed Transpilation Search — Unrolled (T=1) — IBM Boston

Heavy multi-seed transpilation, factored out of `end_to_end_unrolled_boston_t1.ipynb` so it doesn't have to rerun every time the experiment notebook is reset.

**For each `N ∈ {3, 5, 7, 9, 11}` and each unrolled path:**
1. Save a PNG of the path's *pre-transpilation* (logical) circuit.
2. Transpile `TRANSPILE_SEEDS` times against `ibm_boston` at `optimization_level=3`, picking the seed that minimizes 2Q depth (tiebreak: lower 2Q gate count).
3. Save the winning transpiled circuit as a QPY file and a PNG.
4. Append metadata (best seed, depths, gate counts, all-seed stats) to a per-N `summary.json`.

Output layout under `data/transpilations/end_to_end_unrolled_boston_t1/`:
```
n3/
  path_0_pre.png  path_0_post.png  path_0.qpy
  path_1_pre.png  …
  summary.json
n5/  …
```

**Resumability:** every path is checkpointed to `summary.json` immediately after it finishes. Re-running the cell skips paths whose QPY already exists, so kernel deaths or partial runs cost at most one path's work.

**Keep the parameters below in sync** with `end_to_end_unrolled_boston_t1.ipynb` — that notebook reads from this directory and trusts that the same `K`, `T`, `TRANSPILE_SEEDS`, `OPT_LEVEL`, `DEVICE`, `NO_RESET`, and `N_VALUES` were used here.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from qiskit import transpile, qpy

from core.circuit_factory import CircuitFactory
from execution.backend_handler import IBMRuntimeHandler

print('Imports loaded.')

## Step 1: Configuration  (must match `end_to_end_unrolled_boston_t1.ipynb`)

In [ ]:
K = 2                        # Qubits per register
T = 1                        # QPA trial rounds
N_VALUES = [3, 5, 7, 9, 11]
TRANSPILE_SEEDS = 10         # Multi-seed transpilation, pick lowest 2Q depth
OPT_LEVEL = 3                # Transpiler optimization level
DEVICE = 'ibm_boston'
NO_RESET = False

TRANSPILE_DIR = os.path.join('data', 'transpilations', 'end_to_end_unrolled_boston_t1')
os.makedirs(TRANSPILE_DIR, exist_ok=True)

print(f'K={K}, T={T}, N_VALUES={N_VALUES}')
print(f'TRANSPILE_SEEDS={TRANSPILE_SEEDS}, OPT_LEVEL={OPT_LEVEL}, DEVICE={DEVICE!r}')
print(f'Output directory: {TRANSPILE_DIR}')

## Step 2: Backend Setup

In [ ]:
backend_handler = IBMRuntimeHandler(backend_name=DEVICE)
backend = backend_handler.get_backend()

print(f'Backend: {backend.name}')
print(f'Number of qubits: {backend.num_qubits}')

## Step 3: Helpers

Per-N output directory, summary load/save, image save, and the multi-seed picker (same logic as the original `transpile_best_of`).

In [ ]:
def n_dir(n):
    d = os.path.join(TRANSPILE_DIR, f'n{n}')
    os.makedirs(d, exist_ok=True)
    return d

def summary_file(n):
    return os.path.join(n_dir(n), 'summary.json')

def load_summary(n):
    p = summary_file(n)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return {
        'n': n, 'k': K, 't': T,
        'transpile_seeds': TRANSPILE_SEEDS,
        'opt_level': OPT_LEVEL,
        'backend': backend.name,
        'no_reset': NO_RESET,
        'paths': {},
    }

def save_summary(n, summary):
    with open(summary_file(n), 'w') as f:
        json.dump(summary, f, indent=2)

def save_circuit_image(qc, path, fold=120):
    """Save a matplotlib rendering of qc to path. Hides idle wires for readability."""
    fig = qc.draw('mpl', fold=fold, idle_wires=False)
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)

def count_2q_gates(qc):
    return sum(1 for instr in qc.data if len(instr.qubits) == 2)

def transpile_best_of(qc, backend, n_seeds, opt_level):
    """Run n_seeds transpilations of qc; keep the one with lowest 2Q depth (tiebreak: 2Q gate count)."""
    best = None
    seed_stats = []
    for seed in range(n_seeds):
        qc_tr = transpile(qc, backend=backend, optimization_level=opt_level, seed_transpiler=seed)
        d2q = qc_tr.depth(lambda instr: len(instr.qubits) == 2)
        g2q = count_2q_gates(qc_tr)
        seed_stats.append({'seed': seed, '2q_depth': d2q, '2q_gates': g2q})
        if (best is None) or (d2q < best['d2q']) or (d2q == best['d2q'] and g2q < best['g2q']):
            best = {'seed': seed, 'd2q': d2q, 'g2q': g2q, 'circuit': qc_tr}
    return best, seed_stats

print('Helpers defined.')

## Step 4: Main loop — find best seed for every path of every N

For each N: build the unrolled circuits, save the pre-transpilation image, run multi-seed transpilation, save QPY + post-transpilation image, and append metadata to `summary.json`. Skips any path whose QPY file already exists.

In [ ]:
for n in N_VALUES:
    print(f"\n{'='*60}\nN = {n}\n{'='*60}")

    strategy = CircuitFactory.create_strategy('unrolled', K, T, n, no_reset=NO_RESET)
    strategy.set_noise_strategy(None)
    golden_data = strategy.build(0.0)

    nd = n_dir(n)
    summary = load_summary(n)
    summary.setdefault('paths', {})
    print(f'  {len(golden_data)} paths -> {nd}')

    for i, item in enumerate(tqdm(golden_data, desc=f'N={n}', leave=False)):
        qc = item['circuit']
        path_name = item.get('metadata', {}).get('path_name', f'path_{i}')
        qpy_path  = os.path.join(nd, f'{path_name}.qpy')
        pre_img   = os.path.join(nd, f'{path_name}_pre.png')
        post_img  = os.path.join(nd, f'{path_name}_post.png')

        if os.path.exists(qpy_path) and path_name in summary['paths']:
            print(f'  {path_name}: cached (qpy + summary exist) — skipping')
            continue

        try:
            save_circuit_image(qc, pre_img)
        except Exception as e:
            print(f'  WARN: failed to render pre-image for {path_name}: {e}')

        best, seed_stats = transpile_best_of(qc, backend, TRANSPILE_SEEDS, OPT_LEVEL)

        with open(qpy_path, 'wb') as f:
            qpy.dump(best['circuit'], f)

        try:
            save_circuit_image(best['circuit'], post_img)
        except Exception as e:
            print(f'  WARN: failed to render post-image for {path_name}: {e}')

        summary['paths'][path_name] = {
            'best_seed': best['seed'],
            'best_2q_depth': best['d2q'],
            'best_2q_gates': best['g2q'],
            'depth': best['circuit'].depth(),
            'gate_counts': dict(best['circuit'].count_ops()),
            'all_seed_stats': seed_stats,
            'qpy_file': os.path.basename(qpy_path),
            'pre_image': os.path.basename(pre_img),
            'post_image': os.path.basename(post_img),
        }
        save_summary(n, summary)

        ops = dict(best['circuit'].count_ops())
        print(f"  {path_name}: seed={best['seed']}, 2Q depth={best['d2q']}, 2Q gates={best['g2q']}, ops={ops}")

    print(f'\nN={n} complete -> {summary_file(n)}')

## Step 5: Verify all expected outputs landed on disk

Sanity check: enumerate per-N path counts and confirm every path has a QPY, a pre-image, a post-image, and a summary entry.

In [ ]:
report_rows = []
for n in N_VALUES:
    nd = n_dir(n)
    sf = summary_file(n)
    if not os.path.exists(sf):
        report_rows.append({'N': n, 'paths': 0, 'qpy_files': 0, 'pre_imgs': 0, 'post_imgs': 0, 'note': 'no summary.json'})
        continue
    with open(sf) as f:
        summary = json.load(f)
    paths = summary.get('paths', {})
    qpy_count  = sum(os.path.exists(os.path.join(nd, p['qpy_file']))   for p in paths.values())
    pre_count  = sum(os.path.exists(os.path.join(nd, p['pre_image']))  for p in paths.values())
    post_count = sum(os.path.exists(os.path.join(nd, p['post_image'])) for p in paths.values())
    expected = 2 ** ((n - 1) // 2)
    note = 'OK' if (len(paths) == qpy_count == pre_count == post_count == expected) else 'MISMATCH'
    report_rows.append({'N': n, 'paths': len(paths), 'expected': expected,
                        'qpy_files': qpy_count, 'pre_imgs': pre_count, 'post_imgs': post_count,
                        'note': note})

import pandas as pd
print(pd.DataFrame(report_rows).to_string(index=False))